In [1]:
import os
import sys
import pandas as pd
import datetime as dt
from datetime import datetime
from dataretrieval import nwis
from dataretrieval import waterdata
import numpy as np
import requests
import io
import torch
from scripts import data,training
import joblib

In [2]:
train1 = "10132000"
lat1 = 40.96772452
lon1 = -111.437699

train2 = "10136600"
lat2 = 41.13708333
lon2 = -111.9195556

train3 = "10137000"
lat3 = 41.223269
lon3 = -111.988117

test = "10136500" #test streamgage in the middle to avoid boundary effects 
lat_test = 41.1368878
lon_test = -111.8324384

train_ID = [train1,train2,train3]

start = "2016-01-01"
end="2025-12-31"
years= ",".join(str(y) for y in range(datetime.strptime(start, "%Y-%m-%d").year, #daymet didn't like dates for these gages, had to use years parameter!
                                       datetime.strptime(end, "%Y-%m-%d").year + 1))

Note that I reran my Homework 2 for each of the streamgages above to obtain the data files for SWE, please view the code in "https://github.com/eburgon2/Homework2" if you would like specific sytnax for how SWE values were obtained 

for my memory: 
train1 -> 330,392,393,763
train2 -> same, and 896,533,1145,1118,684,814
train3 -> same as 2
test -> same as 2



In [3]:
#only station 1 has less stations, we will include nan columns to fit the data to 10 in the LSTM
swe_stations_1 = ['330','392','393','763']
swe_stations_else = swe_stations_1 + ['533','684','814','896','1118','1145']

swe_1 = data.swe_set(swe_stations_1,train1)
swe_2 = data.swe_set(swe_stations_else,train2)
swe_3 = data.swe_set(swe_stations_else,train3)
swe_test = data.swe_set(swe_stations_else,test)

In [4]:
Q1 = data.discharge(train1,start,end)
Q2 = data.discharge(train2,start,end)
Q3 = data.discharge(train3,start,end)
Q_test = data.discharge(test,start,end)

In [5]:
daymet1 = data.normalize_daymet(data.daymet(train1,lat1,lon1,years))
daymet2 = data.normalize_daymet(data.daymet(train2,lat2,lon2,years))
daymet3 = data.normalize_daymet(data.daymet(train3,lat3,lon3,years))
daymet_Test = data.normalize_daymet(data.daymet(test,lat_test,lon_test,years))

The last data set we can consider is catchment characteristics. However, this isn't really something we can show over time, as they usually don't change drastically quickly (~3 years), so it likely won't be able to train the model well. I also looked through earth access options, and they mostly only give annual data. For this reason, we will be considering only the parameters we have shown above to avoid mistraining the model. 

In [6]:
#now masking the swe_1 to make sure it matches pytorch sizes
#note that the mask method will allow the LSTM to ignore NAN (aka won't break) 
swe_1 = swe_1.reindex(columns=list(swe_2.columns)) #creating Nans, we will mask on the datafram in the model portion of the assignment

training1 = data.full_data(daymet1,swe_1,train1)
training2 = data.full_data(daymet2,swe_2,train2)
training3 = data.full_data(daymet3,swe_3,train3)

testing = data.full_data(daymet_Test,swe_test,test)

In [7]:
#prep for splitting by year
training1['year'] = pd.to_datetime(training1['Date']).dt.year
training2['year'] = pd.to_datetime(training2['Date']).dt.year
training3['year'] = pd.to_datetime(training3['Date']).dt.year

#80/20 split
training_start =2016
training_end =2023
validate_start =2024
validate_end =2025

#develop training and validation input sets for each site
(X_training_1, X_training_2,X_training_3,
 X_validate_1,X_validate_2, X_validate_3) = training.train_validation_split(training1,
                                                                            training2,
                                                                            training3,
                                                                            training_end,
                                                                            validate_start,
                                                                            validate_end,
                                                                            'input')


#develop training and validation output sets for each site
(Y_training_1, Y_training_2,Y_training_3,
 Y_validate_1,Y_validate_2, Y_validate_3) = training.train_validation_split(Q1,
                                                                            Q2,
                                                                            Q3,
                                                                            training_end,
                                                                            validate_start,
                                                                            validate_end,
                                                                            "target")

print(f"Training Rows: Site 1 - {len(X_training_1)}, Site 2 = {len(X_training_2)}, Site 3 - {len(X_training_3)}")
print(f"Validation Rows: Site 1 - {len(X_validate_1)}, Site 2 = {len(X_validate_2)}, Site 3 - {len(X_validate_3)}")
print(f'Percent Validation: {(len(X_validate_1)/(len(X_training_1)+len(X_validate_1)) * 100):.0f}%')

Training Rows: Site 1 - 2922, Site 2 = 2922, Site 3 - 2922
Validation Rows: Site 1 - 731, Site 2 = 731, Site 3 - 731
Percent Validation: 20%


In [8]:
X_t1, mask_t1 = training.masking(X_training_1)
X_t2, mask_t2 = training.masking(X_training_2)
X_t3, mask_t3 = training.masking(X_training_3)

X_v1, mask_v1 = training.masking(X_validate_1)
X_v2, mask_v2 = training.masking(X_validate_2)
X_v3, mask_v3 = training.masking(X_validate_3)

#and for the test set, first have to fix the dataframes to not have the date index
X_test = testing.drop(columns=["Date"]).copy()
Y_test = Q_test.drop(columns=["Date"]).copy()

X_test, Mask_test = training.masking(X_test)

In [9]:
(X_t1_norm, X_t2_norm,X_t3_norm,
X_v1_norm,X_v2_norm,X_v3_norm, xscale) = training.minmax(X_t1, X_t2,X_t3,
                                                         X_v1,X_v2,X_v3,'x')

(Y_t1_norm, Y_t2_norm,Y_t3_norm,
Y_v1_norm,Y_v2_norm,Y_v3_norm,yscale) = training.minmax(Y_training_1, Y_training_2,Y_training_3,
                                                        Y_validate_1,Y_validate_2, Y_validate_3,'y')

#normalize test with training scales
X_test = xscale.transform(X_test)
Y_test = yscale.transform(Y_test)
joblib.dump(yscale, "LSTM Files/yscale.pkl")
joblib.dump(xscale, "LSTM Files/xscale.pkl")

/opt/homebrew/Caskroom/miniforge/base/envs/torchenv/lib/python3.10/site-packages/sklearn/base.py:458: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


['LSTM Files/xscale.pkl']

In [10]:
X_train, M_train, Y_train = training.build_full_sets([X_t1_norm, X_t2_norm,X_t3_norm],[mask_t1,mask_t2,mask_t3],[Y_t1_norm, Y_t2_norm,Y_t3_norm])

X_valid, M_valid, Y_valid = training.build_full_sets([X_v1_norm,X_v2_norm,X_v3_norm],[mask_v1,mask_v2,mask_v3],[Y_v1_norm,Y_v2_norm,Y_v3_norm])

X_test, M_test, Y_test = training.build_full_sets([X_test],[Mask_test],[Y_test])

/Users/lizzieburgon/Hydroinformatics/Homework3/scripts/training.py:91: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1729647065806/work/torch/csrc/utils/tensor_new.cpp:281.)
  xset = torch.tensor(xset, dtype=torch.float32)


In [11]:
#Save so the other notebook file can pull the data

#------Training------
torch.save(X_train, "LSTM Files/Tensors/X_train.pt")
torch.save(M_train, "LSTM Files/Tensors/M_train.pt")
torch.save(Y_train, "LSTM Files/Tensors/Y_train.pt")

#------Validation------
torch.save(X_valid, "LSTM Files/Tensors/X_valid.pt")
torch.save(M_valid, "LSTM Files/Tensors/M_valid.pt")
torch.save(Y_valid, "LSTM Files/Tensors/Y_valid.pt")

#------Testing------
torch.save(X_test, "LSTM Files/Tensors/X_test.pt")
torch.save(M_test, "LSTM Files/Tensors/M_test.pt")
torch.save(Y_test, "LSTM Files/Tensors/Y_test.pt")